# Positive-feature Sketches for Sinkhorn Plans

**Credit.** This notebook is part of *Optimal Transport for Machine Learners* by [Gabriel Peyré](https://www.gpeyre.com/), CNRS and École normale supérieure, PSL University. The book and reproducible sources are available at [github.com/gpeyre/ot4ml](https://github.com/gpeyre/ot4ml).

**Copyright.** Copyright (c) 2025 Gabriel Peyré. Released under the MIT License; see the repository `LICENSE` file.

This notebook generates `fig:sinkhorn-positive-feature-sketching`.  It compares the dense entropic transport plan between two one-dimensional Gaussian mixtures with positive-feature approximations of the Gaussian Gibbs kernel,
$$
    k_\varepsilon(x,y)=\exp\left(-\frac{|x-y|^2}{\varepsilon}\right).
$$
The point is not to benchmark a solver.  The figure shows the actual objects being changed by sketching: the Sinkhorn coupling computed with the sketched positive kernel $\widetilde K=\Phi_X\Phi_Y^\top$, and the associated effective cost
$$
    \widetilde c_\varepsilon(x,y)=-\varepsilon\log \widetilde k_\varepsilon(x,y).
$$
A good sketch is not merely a low-rank approximation of a matrix; it should approximate the kernel accurately enough in logarithmic scale so that $\widetilde c_\varepsilon$ stays close to the quadratic cost $|x-y|^2$.

In [ ]:
# --- OT4ML_COLAB_BOOTSTRAP: standalone local/Colab setup.
from pathlib import Path as _OT4ML_Path
import importlib.util as _OT4ML_importlib_util
import os as _OT4ML_os
import subprocess as _OT4ML_subprocess
import sys as _OT4ML_sys
import urllib.request as _OT4ML_urllib_request


def _ot4ml_find_repo_root():
    """Locate a checkout containing the shared figure-style module.

    Returns:
        Repository root when a checkout is available, otherwise ``None``.
    """
    here = _OT4ML_Path.cwd().resolve()
    for base in (here, *here.parents):
        if (base / "notebooks-figures" / "figure_style.py").exists():
            return base
        if base.name == "notebooks-figures" and (base / "figure_style.py").exists():
            return base.parent
    return None


def _ot4ml_ensure_package(module_name, package_name):
    """Install one missing dependency in the active Colab environment.

    Args:
        module_name: Importable Python module to test.
        package_name: Distribution name passed to ``pip`` when needed.

    Returns:
        ``None``.
    """
    if _OT4ML_importlib_util.find_spec(module_name) is None:
        _OT4ML_subprocess.run(
            [_OT4ML_sys.executable, "-m", "pip", "install", "-q", package_name],
            check=True,
        )


def _ot4ml_download(relative_path, target):
    """Download one small shared file without cloning the full repository.

    Args:
        relative_path: Repository-relative path on the OT4ML website.
        target: Local destination path.

    Returns:
        ``None``.
    """
    if target.exists():
        return
    target.parent.mkdir(parents=True, exist_ok=True)
    bases = (
        "https://raw.githubusercontent.com/gpeyre/ot4ml/main",
        "https://www.gpeyre.com/ot4ml",
    )
    errors = []
    temporary = target.with_suffix(target.suffix + ".tmp")
    for base in bases:
        url = f"{base}/{relative_path}"
        try:
            with _OT4ML_urllib_request.urlopen(url, timeout=30) as response:
                temporary.write_bytes(response.read())
            temporary.replace(target)
            return
        except Exception as error:
            errors.append(f"{url}: {error}")
            temporary.unlink(missing_ok=True)
    details = "\n".join(errors)
    raise RuntimeError(f"Unable to download {relative_path}.\n{details}")


try:
    _OT4ML_IN_COLAB = _OT4ML_importlib_util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    _OT4ML_IN_COLAB = False

_OT4ML_MPLCONFIG = _OT4ML_Path(
    _OT4ML_os.environ.get(
        "MPLCONFIGDIR",
        _OT4ML_Path(_OT4ML_os.environ.get("TMPDIR", "/tmp")) / "mpl-ot4ml",
    )
)
_OT4ML_MPLCONFIG.mkdir(parents=True, exist_ok=True)
_OT4ML_os.environ["MPLCONFIGDIR"] = str(_OT4ML_MPLCONFIG)

_OT4ML_PACKAGES = (
    ('numpy', 'numpy'),
    ('matplotlib', 'matplotlib'),
)
if _OT4ML_IN_COLAB:
    for _module_name, _package_name in _OT4ML_PACKAGES:
        _ot4ml_ensure_package(_module_name, _package_name)

ROOT = _ot4ml_find_repo_root()
if ROOT is None:
    ROOT = (
        _OT4ML_Path("/content/ot4ml-figure-runtime")
        if _OT4ML_IN_COLAB
        else _OT4ML_Path.cwd() / "ot4ml-figure-runtime"
    )
ROOT = ROOT.resolve()

_OT4ML_REQUIRED_FILES = (
    'notebooks-figures/thumbnails/sinkhorn-positive-feature-sketching.png',
)
_style_path = ROOT / "notebooks-figures" / "figure_style.py"
_ot4ml_download("notebooks-figures/figure_style.py", _style_path)
for _relative_path in _OT4ML_REQUIRED_FILES:
    _ot4ml_download(_relative_path, ROOT / _relative_path)

(ROOT / "OT4ML" / "figures").mkdir(parents=True, exist_ok=True)
(ROOT / "notebooks-figures" / "thumbnails").mkdir(parents=True, exist_ok=True)
_OT4ML_os.chdir(ROOT)
_figures_path = str(ROOT / "notebooks-figures")
if _figures_path not in _OT4ML_sys.path:
    _OT4ML_sys.path.insert(0, _figures_path)


In [1]:
from pathlib import Path
import os
import sys


for candidate in [Path.cwd(), Path.cwd() / "notebooks-figures", Path.cwd().parent / "notebooks-figures"]:
    if (candidate / "figure_style.py").exists():
        sys.path.insert(0, str(candidate.resolve()))
        break
else:
    raise RuntimeError("Could not locate figure_style.py")

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LinearSegmentedColormap
from numpy.polynomial.hermite import hermgauss

from figure_style import (
    BLUE,
    RED,
    VIOLET,
    GRAY,
    ROOT,
    figure_dir,
    save_pdf,
    setup_matplotlib,
)

setup_matplotlib()

NAME = "sinkhorn-positive-feature-sketching"
OUT = figure_dir(NAME)
THUMB = ROOT / "notebooks-figures" / "thumbnails"
ARXIV_OUT = ROOT / "arxiv" / "figures"
THUMB.mkdir(parents=True, exist_ok=True)
ARXIV_OUT.mkdir(parents=True, exist_ok=True)

# Remove obsolete panels from older versions of this figure.
for stale_panel in [OUT / "rank-001.pdf", ARXIV_OUT / f"{NAME}--rank-001.pdf"]:
    stale_panel.unlink(missing_ok=True)

RNG = np.random.default_rng(20260704)

## Mixtures and Exact Sinkhorn Plan

The source and target are discretized on a common one-dimensional grid.  The entropic plan is
$$
    P_\varepsilon=\operatorname{diag}(u)K\operatorname{diag}(v),
    \qquad u=\frac{a}{Kv},\quad v=\frac{b}{K^\top u}.
$$
All divisions are componentwise.  The grid is small enough that the dense plan can be used as a visual reference.

In [2]:
def gaussian(x, mean, sigma):
    """Evaluate a Gaussian profile on the supplied grid.
    
    Args:
        x: Evaluation points, source samples, or horizontal grid coordinates.
        mean: Mean vector or scalar mean of a Gaussian component.
        sigma: Standard deviation, covariance scale, or diffusion strength.
    
    Returns:
        Gaussian profile values evaluated on the supplied grid.
    """
    return np.exp(-0.5 * ((x - mean) / sigma) ** 2) / (sigma * np.sqrt(2.0 * np.pi))


def normalize(hist):
    """Normalize an array to a stable probability or display scale.
    
    Args:
        hist: Histogram values.
    
    Returns:
        Normalized an array to a stable probability or display scale.
    """
    hist = np.maximum(np.asarray(hist, dtype=float), 0.0)
    return hist / hist.sum()


n = 150
x = np.linspace(-2.2, 2.2, n)
y = x.copy()

a = normalize(0.55 * gaussian(x, -0.85, 0.28) + 0.45 * gaussian(x, 0.72, 0.20))
b = normalize(
    0.34 * gaussian(y, -1.15, 0.22)
    + 0.42 * gaussian(y, 0.18, 0.30)
    + 0.24 * gaussian(y, 1.05, 0.18)
)

# The cost is |x-y|^2, hence the Gibbs kernel is a Gaussian kernel.
epsilon = 0.020
C = (x[:, None] - y[None, :]) ** 2
K = np.exp(-C / epsilon)


def sinkhorn_kernel(K, a, b, *, n_iter=20_000, tol=1e-12):
    """Compute a Sinkhorn coupling from the exact kernel matrix.
    
    Args:
        K: Kernel matrix or transition matrix.
        a: Source histogram weights or scalar coefficient, depending on context.
        b: Target histogram weights or scalar coefficient, depending on context.
        n_iter: Number of fixed-point or optimization iterations.
        tol: Numerical tolerance used as a stopping or filtering criterion.
    
    Returns:
        Sinkhorn coupling computed from the full kernel matrix.
    """
    u = np.ones_like(a)
    v = np.ones_like(b)
    tiny = 1e-300
    err = np.inf
    for it in range(n_iter):
        u = a / (K @ v + tiny)
        v = b / (K.T @ u + tiny)
        if it % 200 == 0 and it > 0:
            P = (u[:, None] * K) * v[None, :]
            err = max(np.abs(P.sum(axis=1) - a).max(), np.abs(P.sum(axis=0) - b).max())
            if err < tol:
                break
    P = (u[:, None] * K) * v[None, :]
    return u, v, P, err, it


_, _, P_exact, err_exact, it_exact = sinkhorn_kernel(K, a, b)
err_exact, it_exact, P_exact.sum()


(np.float64(3.844771723215956e-13), 1800, np.float64(1.0))

## Positive Exponential Features

Write $q_x=\sqrt{2}(x-c)/\sqrt{\varepsilon}$, where the centering constant $c$ is arbitrary for this translation-invariant kernel.  For $\omega\sim\mathcal N(0,1)$, define
$$
    \psi_\omega(q)=\exp\left(\omega q-q^2\right).
$$
Then
$$
    \mathbb E\,\psi_\omega(q_x)\psi_\omega(q_y)
    =\exp\left(-\frac12(q_x-q_y)^2\right)
    =k_\varepsilon(x,y).
$$
The book discusses the Monte-Carlo version.  For a deterministic and reproducible pedagogical picture, the notebook uses the same positive feature identity but replaces the Gaussian expectation by Gauss--Hermite quadrature.  A large quadrature rank gives a faithful positive sketch, while a coarse rank shows the geometric distortion caused by an overly compressed positive kernel.

In [3]:
def positive_gaussian_features(x, y, epsilon, rank):
    """Build positive quadrature features for the Gaussian kernel.
    
    Args:
        x: Evaluation points, source samples, or horizontal grid coordinates.
        y: Target samples or vertical grid coordinates.
        epsilon: Entropic or numerical regularization strength.
        rank: Rank or number of latent factors in the approximation.
    
    Returns:
        Pair of nonnegative feature matrices whose product approximates the Gaussian kernel.
    """
    nodes, weights = hermgauss(rank)
    # Convert Hermite quadrature for exp(-z^2) dz to expectation under N(0, 1).
    omega = np.sqrt(2.0) * nodes
    weights = weights / np.sqrt(np.pi)

    # Centering is harmless for the translation-invariant kernel and improves conditioning.
    center = 0.5 * (x.mean() + y.mean())
    qx = np.sqrt(2.0) * (x - center) / np.sqrt(epsilon)
    qy = np.sqrt(2.0) * (y - center) / np.sqrt(epsilon)

    PhiX = np.sqrt(weights)[None, :] * np.exp(qx[:, None] * omega[None, :] - qx[:, None] ** 2)
    PhiY = np.sqrt(weights)[None, :] * np.exp(qy[:, None] * omega[None, :] - qy[:, None] ** 2)
    return PhiX, PhiY


def sinkhorn_features(PhiX, PhiY, a, b, *, n_iter=20_000, tol=1e-12):
    """Compute a Sinkhorn coupling from positive kernel features.
    
    Args:
        PhiX: Feature matrix evaluated on the source samples.
        PhiY: Feature matrix evaluated on the target samples.
        a: Source histogram weights or scalar coefficient, depending on context.
        b: Target histogram weights or scalar coefficient, depending on context.
        n_iter: Number of fixed-point or optimization iterations.
        tol: Numerical tolerance used as a stopping or filtering criterion.
    
    Returns:
        Sinkhorn coupling computed from the positive feature approximation.
    """
    u = np.ones_like(a)
    v = np.ones_like(b)
    tiny = 1e-300
    err = np.inf
    for it in range(n_iter):
        u = a / (PhiX @ (PhiY.T @ v) + tiny)
        v = b / (PhiY @ (PhiX.T @ u) + tiny)
        if it % 500 == 0 and it > 0:
            P = (u[:, None] * PhiX) @ (v[:, None] * PhiY).T
            err = max(np.abs(P.sum(axis=1) - a).max(), np.abs(P.sum(axis=0) - b).max())
            if err < tol:
                break
    P = (u[:, None] * PhiX) @ (v[:, None] * PhiY).T
    return u, v, P, err, it


def weighted_quantile(values, weights, quantile):
    """Compute quantiles of a discretized weighted one-dimensional law.
    
    Args:
        values: Array of scalar values to integrate, normalize, transform, or display.
        weights: Probability weights or mixture weights.
        quantile: Quantile level or quantile curve.
    
    Returns:
        Computed quantile values from the discretized one-dimensional law.
    """
    order = np.argsort(values)
    values = values[order]
    weights = weights[order]
    cdf = np.cumsum(weights)
    cdf = cdf / cdf[-1]
    return float(np.interp(quantile, cdf, values))


def kernel_plan_diagnostics(K_sketch, P_sketch):
    """Compute diagnostics for the sketched Sinkhorn kernel and plan.
    
    Args:
        K_sketch: Sketched Sinkhorn kernel approximation.
        P_sketch: Transport plan obtained with the sketched kernel.
    
    Returns:
        Kernel plan diagnostics.
    """
    log_error = np.log(K_sketch + 1e-300) - np.log(K + 1e-300)
    abs_log_error = np.abs(log_error).ravel()
    plan_weights = P_exact.ravel()
    return {
        "median |log K error|": float(np.median(abs_log_error)),
        "P-weighted 99% |log K error|": weighted_quantile(abs_log_error, plan_weights, 0.99),
        "P L1 error": float(np.sum(np.abs(P_sketch - P_exact))),
        "P weighted log RMSE": float(np.sqrt(np.sum(P_exact * log_error**2))),
        "max effective cost error on display": float(np.max(np.abs(-epsilon * np.log(K_sketch + 1e-300) - C))),
    }


ranks = {"rank40": 40, "rank10": 10, "rank3": 3}
plans = {"exact": P_exact}
kernels = {"exact": K}
diagnostics = {
    "exact": {
        "marginal residual": float(err_exact),
        "iterations": int(it_exact),
        "P L1 error": 0.0,
        "P weighted log RMSE": 0.0,
        "max effective cost error on display": 0.0,
    }
}
for key, rank in ranks.items():
    PhiX, PhiY = positive_gaussian_features(x, y, epsilon, rank)
    K_sketch = PhiX @ PhiY.T
    _, _, P_sketch, marginal_error, n_iter = sinkhorn_features(PhiX, PhiY, a, b)
    plans[key] = P_sketch
    kernels[key] = K_sketch
    diagnostics[key] = {
        "rank": rank,
        "marginal residual": float(marginal_error),
        "iterations": int(n_iter),
        **kernel_plan_diagnostics(K_sketch, P_sketch),
    }

diagnostics


{'exact': {'marginal residual': 3.844771723215956e-13,
  'iterations': 1800,
  'P L1 error': 0.0,
  'P weighted log RMSE': 0.0,
  'max effective cost error on display': 0.0},
 'rank40': {'rank': 40,
  'marginal residual': 9.598571937274869e-14,
  'iterations': 1500,
  'median |log K error|': 1.8796511871116346,
  'P-weighted 99% |log K error|': 169.57283782169074,
  'P L1 error': 0.7349448083236992,
  'P weighted log RMSE': 48.93300901190946,
  'max effective cost error on display': 10.608932936775025},
 'rank10': {'rank': 10,
  'marginal residual': 8.555656183517613e-15,
  'iterations': 1000,
  'median |log K error|': 33.62678166506424,
  'P-weighted 99% |log K error|': 312.1988215601,
  'P L1 error': 1.177591070896307,
  'P weighted log RMSE': 112.65359730854321,
  'max effective cost error on display': 13.815510557964274},
 'rank3': {'rank': 3,
  'marginal residual': 1.5612511283791264e-17,
  'iterations': 500,
  'median |log K error|': 63.69949767158394,
  'P-weighted 99% |log K er

## Rendering

The first row displays the coupling as a purple density in the $(x,y)$ plane.  The red curve is the source marginal and the blue curve is the target marginal, placed along the two axes so that the reader can see that every displayed coupling has the same prescribed marginals.  The second row displays the effective cost $\widetilde c_\varepsilon=-\varepsilon\log\widetilde k_\varepsilon$ for the same kernel approximation.  Black level sets make it easy to compare each sketch with the quadratic cost $|x-y|^2$.

In [ ]:
plan_cmap = LinearSegmentedColormap.from_list(
    "ot4ml_plan",
    ["#ffffff", "#f5eff8", "#d9c7e2", "#9c6fb0", VIOLET, "#351047"],
)

cost_cmap = LinearSegmentedColormap.from_list(
    "ot4ml_cost",
    ["#ffffff", "#f7f9fc", "#dce6f1", "#a8b8cf", "#657892"],
)


def panel_geometry():
    """Compute panel geometry.
    
    Returns:
        Panel geometry values.
    """
    xmin, xmax = x.min(), x.max()
    ymin, ymax = y.min(), y.max()
    span_x = xmax - xmin
    span_y = ymax - ymin
    gap_x = 0.18 * span_x
    gap_y = 0.18 * span_y
    curve_h = 0.14 * span_y
    curve_w = 0.14 * span_x
    return xmin, xmax, ymin, ymax, gap_x, gap_y, curve_w, curve_h


def style_panel(ax, *, with_gutters=True):
    """Compute style panel.
    
    Args:
        ax: Matplotlib axes on which the object is drawn.
        with_gutters: Whether to include gutters between matrix cells.
    
    Returns:
        None. Draws on the provided Matplotlib axes.
    """
    xmin, xmax, ymin, ymax, gap_x, gap_y, _, _ = panel_geometry()
    # Tiny top/right padding prevents PDF renderers from clipping the frame at the canvas edge.
    right_pad = 0.012 * (xmax - xmin)
    top_pad = 0.012 * (ymax - ymin)
    if with_gutters:
        ax.set_xlim(xmin - 1.05 * gap_x, xmax + right_pad)
        ax.set_ylim(ymin - 1.05 * gap_y, ymax + top_pad)
    else:
        ax.set_xlim(xmin, xmax + right_pad)
        ax.set_ylim(ymin, ymax + top_pad)
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)


def draw_domain_box(ax, *, zorder=6):
    """Render the domain box with the OT4ML plotting style.
    
    Args:
        ax: Matplotlib axes on which the object is drawn.
        zorder: Matplotlib drawing order.
    
    Returns:
        None. Draws on the provided Matplotlib axes.
    """
    xmin, xmax, ymin, ymax, *_ = panel_geometry()
    ax.plot(
        [xmin, xmax, xmax, xmin, xmin],
        [ymin, ymin, ymax, ymax, ymin],
        color="#202020",
        lw=0.58,
        solid_capstyle="butt",
        solid_joinstyle="miter",
        clip_on=False,
        zorder=zorder,
    )


def save_panel_pdf(fig, path):
    """Evaluate save panel probability density on the supplied grid.
    
    Args:
        fig: Matplotlib figure used for the display.
        path: Input or output filesystem path.
    
    Returns:
        None. Writes the requested figure or export file to disk.
    """
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.subplots_adjust(left=0, right=1, bottom=0, top=1)
    fig.savefig(path)


### Rendering helpers

The helpers `render_plan`, `effective_cost`, `render_cost` handle the panel layout, style conventions, and file export without hiding the numerical construction. This separation makes the notebook easier to audit: numerical choices appear before final rendering choices.

$$P\mathbf 1=a,\qquad P^\top\mathbf 1=b.$$



In [ ]:


def render_plan(ax, P, *, vmax):
    """Render a transport-plan panel with the OT4ML plotting style.
    
    Args:
        ax: Matplotlib axes on which the object is drawn.
        P: Coupling, transport plan, or transition matrix.
        vmax: Upper color scale used for image normalization.
    
    Returns:
        None. Draws on the provided Matplotlib axes.
    """
    xmin, xmax, ymin, ymax, _, _, curve_w, curve_h = panel_geometry()

    ax.imshow(
        P.T,
        extent=[xmin, xmax, ymin, ymax],
        origin="lower",
        cmap=plan_cmap,
        interpolation="bilinear",
        norm=mpl.colors.PowerNorm(gamma=0.74, vmin=0.0, vmax=vmax),
        aspect="auto",
        zorder=1,
    )

    ax.plot(x, ymin - curve_h * a / a.max(), color=RED, lw=1.25, zorder=5)
    ax.fill_between(x, ymin, ymin - curve_h * a / a.max(), color=RED, alpha=0.12, lw=0)
    ax.plot(xmin - curve_w * b / b.max(), y, color=BLUE, lw=1.25, zorder=5)
    ax.fill_betweenx(y, xmin, xmin - curve_w * b / b.max(), color=BLUE, alpha=0.12, lw=0)

    draw_domain_box(ax)
    style_panel(ax, with_gutters=True)


def effective_cost(K_eff):
    """Assemble the effective cost used by the OT solver.
    
    Args:
        K_eff: Effective kernel matrix reconstructed from the feature sketch.
    
    Returns:
        Effective cost matrix reconstructed from the approximated Sinkhorn kernel.
    """
    return -epsilon * np.log(np.maximum(K_eff, 1e-300))


def render_cost(ax, K_eff, *, vmin, vmax, levels):
    """Render the cost with the OT4ML plotting style.
    
    Args:
        ax: Matplotlib axes on which the object is drawn.
        K_eff: Effective kernel matrix reconstructed from the feature sketch.
        vmin: Lower color-scale bound.
        vmax: Upper color scale used for image normalization.
        levels: Contour levels or quantization levels for rendering.
    
    Returns:
        None. Draws on the provided Matplotlib axes.
    """
    xmin, xmax, ymin, ymax, *_ = panel_geometry()
    C_eff = effective_cost(K_eff)
    C_show = np.clip(C_eff, vmin, vmax)
    ax.imshow(
        C_show.T,
        extent=[xmin, xmax, ymin, ymax],
        origin="lower",
        cmap=cost_cmap,
        interpolation="bilinear",
        vmin=vmin,
        vmax=vmax,
        aspect="auto",
        zorder=1,
    )
    ax.contour(
        x,
        y,
        C_eff.T,
        levels=levels,
        colors="#161616",
        linewidths=0.34,
        alpha=0.58,
        zorder=5,
    )
    ax.contour(
        x,
        y,
        C_eff.T,
        levels=levels[::2],
        colors="#000000",
        linewidths=0.50,
        alpha=0.42,
        zorder=6,
    )
    draw_domain_box(ax, zorder=7)
    # Use the same outer frame as the plan row, so the cost square aligns with the coupling square.
    style_panel(ax, with_gutters=True)


panel_info = [
    ("exact", "exact", "exact"),
    ("rank40", "rank-040", "r=40"),
    ("rank10", "rank-010", "r=10"),
    ("rank3", "rank-003", "r=3"),
]

# Use a common robust color scale: identical across panels, but not dominated by a single pixel.
all_positive = np.concatenate([plans[key][plans[key] > 0].ravel() for key, _, _ in panel_info])
vmax_plan = float(np.quantile(all_positive, 0.9975))


### Panel generation

The next cell iterates over the selected times, parameters, or panels and writes the PDF files used by the book.  The mathematical solve has already been encoded above; this block mainly controls reproducible rendering.


In [4]:

# The effective-cost panels focus on the region where the Gibbs kernel is not numerically flat.
cost_vmin = 0.0
cost_vmax = float(np.quantile(C, 0.70))
cost_levels = np.linspace(0.10, cost_vmax, 8)

for key, filename, _ in panel_info:
    fig, ax = plt.subplots(figsize=(2.30, 2.10))
    render_plan(ax, plans[key], vmax=vmax_plan)
    save_panel_pdf(fig, OUT / f"{filename}.pdf")
    save_panel_pdf(fig, ARXIV_OUT / f"{NAME}--{filename}.pdf")
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(2.30, 2.10))
    render_cost(ax, kernels[key], vmin=cost_vmin, vmax=cost_vmax, levels=cost_levels)
    save_panel_pdf(fig, OUT / f"cost-{filename}.pdf")
    save_panel_pdf(fig, ARXIV_OUT / f"{NAME}--cost-{filename}.pdf")
    plt.close(fig)

# Thumbnail/contact sheet for GitHub and MyST.
fig, axes = plt.subplots(2, 4, figsize=(9.6, 4.15))
for ax, (key, _, _) in zip(axes[0], panel_info):
    render_plan(ax, plans[key], vmax=vmax_plan)
for ax, (key, _, _) in zip(axes[1], panel_info):
    render_cost(ax, kernels[key], vmin=cost_vmin, vmax=cost_vmax, levels=cost_levels)
fig.subplots_adjust(wspace=0.015, hspace=0.025, left=0.01, right=0.99, bottom=0.02, top=0.98)
fig.savefig(THUMB / f"{NAME}.png", dpi=190)
plt.close(fig)

OUT, [f"{filename}.pdf" for _, filename, _ in panel_info], [f"cost-{filename}.pdf" for _, filename, _ in panel_info], THUMB / f"{NAME}.png"


(PosixPath('/Users/gpeyre/Dropbox/github/ot4ml/OT4ML/figures/sinkhorn-positive-feature-sketching'),
 ['exact.pdf', 'rank-040.pdf', 'rank-010.pdf', 'rank-003.pdf'],
 ['cost-exact.pdf',
  'cost-rank-040.pdf',
  'cost-rank-010.pdf',
  'cost-rank-003.pdf'],
 PosixPath('/Users/gpeyre/Dropbox/github/ot4ml/notebooks-figures/thumbnails/sinkhorn-positive-feature-sketching.png'))